# Alphamind新手入门之三：投资组合构建

alpha-mind的portfolio文件夹提供了构建组合的工具函数。
- 不同的构建方式实质上对应了不同的优化问题。
- 该文件夹报告的构建方式包括线性构建、均值方差构建等。

### 线性构建： *linear_builder*
- 目标: 在线性约束下最大化组合的预期收益率。
- 线性约束包括
  - 标的权重的上下限 *lbound, ubound*
  - 组合风险暴露的上下限 *risk_target* : 函数内部会以因子矩阵乘以权重得到组合的风险暴露。 
  - 仓位调整的换手率上限(换手率以目标权重和当前权重向量的一范数表示，即差的绝对值之和) *turn_over_target*

In [ ]:
# 假设有5个标的
# 假设当前标的的权重
# 假设标的预期收益向量
# 假设标的风险因子矩阵, 仅有一个风险因子
# 约束条件
# 约束条件1 - 标的权重的上下限
# 限制条件2 - 组合风险暴露的上下限
# 限制条件3 - 仓位调整的换手率上限（下限为0，故无需设置）
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6


### 均值方差构建: *mean_variance_builder*
- 根据经典的均值方差组合优化问题构建模型。
- 目标函数为


\begin{align*}
&\mathop{\arg\max}_{\omega} \ \  \omega_a R - \frac{1}{2} \lambda \sigma   \\
&s.t.\quad
\begin{cases}
\omega_L  \le \omega_a \le \omega_U  \\
\sigma_L \le \sigma \le \sigma_U
\end{cases}
\end{align*}




其中
- $\omega_a$是组合的主动权重向量，如果用$\omega$和$\omega_b$表示组合标的绝对权重向量和基准指数的标的权重向量，那么$\omega_a =\omega -\omega_b$。
- $R$是组合标的预期收益率向量。
- $\lambda$是风险厌恶系数, $\sigma$是整个组合收益率的方差。



##### 函数的参数说明和使用

*mean_variance_builder* 接受的参数有
- 组合标的的预期收益率向量 *er* 
- 风险模型 *risk_model* (optional)
- 基准指数的标的权重向量*bm*
- 组合标的的权重上下限 *lbound, ubound*
- 组合风险暴露矩阵 *risk_exposure* (optional)
- 组合风险暴露的上下限 *risk_target* (optional)
- 风险厌恶系数 *lam* (optional)

NOTE 1: 组合标的的权重上下限 *lbound, ubound* 是指绝对权重向量的上下限。
- 代码中会计算主动权重上下限，即 *$\omega_L$=lbound - $\omega_b$*, *$\omega_U$=ubound - $\omega_b$*。最后会把$\omega_b$加回到优化问题的解，得到组合标的最优权重。
- 代码中也会计算主动收益的风险上下限， 即*risk_target_lbound - risk_exposure $\omega_b$*, *risk_target_ubound + risk_exposure $\omega_b$*。

NOTE 2: 为了计算组合的方差，*mean_variance_builder*接受两种输入参数：
- 标的收益率的协方差矩阵$\Sigma$，由此可以计算得到

\begin{equation*}
    \sigma = \omega_a^T\Sigma \omega_a
\end{equation*}

其中$\Sigma$对应入参中的risk_model['cov']。

- 风险模型：因子收益率的协方差矩阵$F$, 特质收益率的协方差矩阵$\Delta$(对角矩阵)
\begin{equation*}
    \sigma = \omega_a^T \left(X F X^T + \Delta \right)  \omega_a
\end{equation*}

由于一般情况下风险因子的数量小于标的股票的数量，所以$\omega_a^T X F X^T \omega_a$的时间复杂度要小于$\omega_a^T\Sigma \omega_a$， 所以第二种方法效率更高。

其中， $F$对应入参中的risk_model['factor_cov'], $\Delta$对应risk_model['idsync']，$X$对应risk_model['factor_loading']

In [ ]:
# 假设有3个标的
# 假设标的预期收益向量
# 假设收益的协方差矩阵
# 假设收益的非系统性方差(Idiosyncratic Risk)
# 最终的协方差矩阵为
# 假设标的风险暴露矩阵, 有两个风险因子
# 假设基准指数的权重向量
# 约束条件
# 约束条件1 - 标的绝对权重的上下限
# 约束条件2 - 组合风险暴露的上下限
# 建立风险模型，假设直接使用标的的协方差矩阵计算组合的方差
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.portfolio.linearbuilder import linear_builder
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder


In [ ]:
from alphamind.portfolio.rankbuilder import rank_build

latest = panel.filter(pc.equal(panel["trade_date"], dates[-1]))
signal = latest["factor"].to_numpy(zero_copy_only=False)
ranked = rank_build(signal, use_rank=3).flatten().astype(float) / 3.0
risk = np.ones((n_assets, 1))
target = (np.array([1.0]), np.array([1.0]))
linear_status, _, linear_weights = linear_builder(signal, 0.0, 1.0, risk, target)


In [ ]:
benchmark = np.full(n_assets, 1.0 / n_assets)
risk_model = {"cov": np.eye(n_assets) * 0.02, "factor_cov": None, "factor_loading": None, "idsync": None}
mv_status, _, mv_weights = mean_variance_builder(
    signal, risk_model, benchmark, 0.0, 1.0, risk, target
)
assert ranked.sum() == 1.0
assert linear_status in ("optimal", "optimal_inaccurate")
assert mv_status in ("optimal", "optimal_inaccurate")
pa.table({"code": latest["code"], "ranked": ranked, "linear": linear_weights, "mv": mv_weights})
